<a href="https://colab.research.google.com/github/vinhlq2512/research-code-lab/blob/feat/tapta-data-scalability/notebooks/tapta_scalability_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 TAPTA: Data Scalability & High-Resource Continual Relation Extraction
### Protocol FewRel (8 tasks × 10 relations), Rehearsal-Free $M=0$, Frozen BERT-base, Seed 2021

Notebook này phục vụ nghiên cứu **Data Scalability Study**: Mở rộng từ Few-Shot (5-shot) sang High-Resource CRE (10, 50, 100, và Full-Data ~420 mẫu/quan hệ).

**Mục tiêu trả lời 5 Research Questions:**
- **RQ1 (Data Scalability):** Hiệu năng TAPTA thay đổi thế nào khi tăng mẫu huấn luyện?
- **RQ2 (Routing Effectiveness):** Tăng dữ liệu có cải thiện độ chính xác của Task Router không?
- **RQ3 (Catastrophic Forgetting):** TAPTA có kháng quên lãng tốt hơn khi dữ liệu task mới tăng?
- **RQ4 (Rehearsal-Free vs. Exemplar-Based):** TAPTA Rehearsal-Free ($M=0$) có cạnh tranh được với các phương pháp replay ($M=10$) như RP-CRE, EMAR, ConPL?
- **RQ5 (Computational & Memory Efficiency):** Chi phí bộ nhớ (RAM, VRAM, Persistent Memory ~1.03 MB) duy trì ổn định ra sao?

## 1. Kiểm tra Môi trường GPU (NVIDIA CUDA Preflight Check)

In [ ]:
import torch
print("=" * 60)
print("CUDA / GPU PREFLIGHT CHECK")
print("=" * 60)
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:     {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:     {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"CUDA Version:   {torch.version.cuda}")
else:
    print("⚠️ CẢNH BÁO: Không tìm thấy GPU CUDA! Vui lòng chọn Runtime -> Change runtime type -> T4 GPU.")
print("=" * 60)

## 2. Clone / Cập nhật Repository (Nhánh `feat/tapta-data-scalability`)

In [ ]:
import os
REPO_URL = "https://github.com/vinhlq2512/research-code-lab.git"
BRANCH = "feat/tapta-data-scalability"
REPO_DIR = "/content/research-code-lab"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
!pip install -q transformers pyyaml scikit-learn tabulate

## 3. Chạy Kiểm Thử Unit Test (Data Leakage & Nested Sampling Check)

In [ ]:
!pytest tests/test_scalable_loader.py -v

## 4. Chạy Thực Nghiệm Scalability (Lựa chọn Cấu hình hoặc Chạy Toàn Bộ)

In [ ]:
#@title Cấu hình Thí nghiệm { run: "auto" }
experiment_choice = "ds05 (5-shot baseline)" #@param ["ds05 (5-shot baseline)", "ds10 (10-shot)", "ds50 (50-shot)", "ds100 (100-shot)", "ds_full (~420-shot Full)", "RUN_ALL_SUITE"]
enable_oracle_diagnostic = True #@param {type:"boolean"}

exp_map = {
    "ds05 (5-shot baseline)": "configs/scalability/p4_fewrel_ds05_seed2021.yaml",
    "ds10 (10-shot)": "configs/scalability/p4_fewrel_ds10_seed2021.yaml",
    "ds50 (50-shot)": "configs/scalability/p4_fewrel_ds50_seed2021.yaml",
    "ds100 (100-shot)": "configs/scalability/p4_fewrel_ds100_seed2021.yaml",
    "ds_full (~420-shot Full)": "configs/scalability/p4_fewrel_ds_full_seed2021.yaml",
}

oracle_flag = "--oracle_diagnostic" if enable_oracle_diagnostic else ""

if experiment_choice == "RUN_ALL_SUITE":
    print("🚀 Đang chạy toàn bộ Suite 5 thực nghiệm (DS-05 -> DS-FULL)...")
    !bash scripts/run_all_scalability.sh cuda
else:
    cfg = exp_map[experiment_choice]
    print(f"🚀 Đang chạy thực nghiệm: {experiment_choice} với {cfg}...")
    !python experiments/run_scalability_study.py --config {cfg} --device cuda {oracle_flag}

## 5. Tổng hợp Bảng Kết quả Scalability & Đối sánh Baselines (RQ1 - RQ5)

In [ ]:
import json
from pathlib import Path
from tabulate import tabulate

exp_keys = ["fewrel_ds05", "fewrel_ds10", "fewrel_ds50", "fewrel_ds100", "fewrel_ds_full"]
rows = []

for exp in exp_keys:
    summary_file = Path(f"runs/tapta_scalability/{exp}/seed_2021/summary.json")
    res_file = Path(f"runs/tapta_scalability/{exp}/seed_2021/resource_usage.json")
    if summary_file.exists():
        with summary_file.open() as f:
            s = json.load(f)
        res = {}
        if res_file.exists():
            with res_file.open() as f:
                res = json.load(f)
        rows.append([
            s.get("experiment_id"),
            s.get("train_shot"),
            f"{s.get('final_decision_fusion_accuracy', 0)*100:.2f}%",
            f"{s.get('average_incremental_accuracy', 0)*100:.2f}%",
            f"{s.get('average_forgetting', 0)*100:.2f}%",
            f"{s.get('router_top1_accuracy', 0)*100:.2f}%",
            f"{s.get('total_training_time_seconds', 0):.1f}s",
            f"{res.get('peak_gpu_memory_mb', 0):.1f} MB",
            f"{res.get('total_persistent_memory_kb', 0):.1f} KB"
        ])

headers = ["Experiment", "Train Shot", "Final Acc", "AIA", "Forgetting", "Router Top-1", "Train Time", "Peak VRAM", "Buffer Size"]
if rows:
    print(tabulate(rows, headers=headers, tablefmt="grid"))
else:
    print("Chưa tìm thấy kết quả trong runs/tapta_scalability/. Hãy chạy Bước 4 trước!")

## 6. Đóng gói & Tải về Kết quả Thực Nghiệm (Artifacts Archive)

In [ ]:
import shutil
from google.colab import files

archive_dir = "runs/tapta_scalability"
zip_name = "/content/tapta_scalability_results"

if os.path.exists(archive_dir):
    shutil.make_archive(zip_name, "zip", archive_dir)
    print(f"Đã đóng gói thành công: {zip_name}.zip")
    files.download(f"{zip_name}.zip")
else:
    print(f"Thư mục {archive_dir} không tồn tại!")